# Automated Surface Scratch Detection (DAGM 2007)
This notebook trains a U-Net model with a ResNet34 encoder for defect detection.

In [ ]:
!pip install segmentation-models-pytorch opencv-python-headless

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import json

In [ ]:
class DAGMDataset(Dataset):
    def __init__(self, root_dir, classes=None, split='Train', transform=None):
        self.root_dir = root_dir
        self.split = split
        self.transform = transform
        self.samples = []
        
        if classes is None:
            classes = [d for d in os.listdir(root_dir) if d.startswith('Class') and os.path.isdir(os.path.join(root_dir, d))]
            
        for cls in classes:
            split_dir = os.path.join(root_dir, cls, split)
            if not os.path.exists(split_dir):
                continue
                
            label_dir = os.path.join(split_dir, 'Label')
            
            for file_name in os.listdir(split_dir):
                if file_name.endswith('.PNG') and not file_name.endswith('_label.PNG'):
                    img_path = os.path.join(split_dir, file_name)
                    base_name = os.path.splitext(file_name)[0]
                    mask_name = f"{base_name}_label.PNG"
                    mask_path = os.path.join(label_dir, mask_name)
                    
                    if not os.path.exists(mask_path):
                        mask_path = None
                        
                    self.samples.append((img_path, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        
        if mask_path is not None:
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            _, mask = cv2.threshold(mask, 127, 1, cv2.THRESH_BINARY)
        else:
            mask = np.zeros_like(image[:,:,0])
            
        mask = np.expand_dims(mask, axis=-1)
        
        image_tensor = torch.from_numpy(image).float().permute(2, 0, 1) / 255.0
        mask_tensor = torch.from_numpy(mask).float().permute(2, 0, 1)
            
        return image_tensor, mask_tensor

In [ ]:
def calculate_metrics(preds, masks, threshold=0.5):
    preds = (preds > threshold).float()
    preds = preds.view(-1)
    masks = masks.view(-1)
    
    tp = (preds * masks).sum().item()
    fp = (preds * (1 - masks)).sum().item()
    fn = ((1 - preds) * masks).sum().item()
    tn = ((1 - preds) * (1 - masks)).sum().item()
    
    accuracy = (tp + tn) / (tp + fp + fn + tn + 1e-7)
    precision = tp / (tp + fp + 1e-7)
    recall = tp / (tp + fn + 1e-7)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-7)
    iou = tp / (tp + fp + fn + 1e-7)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'iou': iou
    }

### Important: Update `root_dir` below based on where you upload the DAGM dataset in Colab (e.g., `/content/DAGM_KaggleUpload`)

In [ ]:
# Hyperparameters
batch_size = 16  # You can increase this on Colab GPU
epochs = 10
learning_rate = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# TODO: Update this path after uploading dataset to Google Colab
root_dir = "/content/DAGM_KaggleUpload"

# Using Class1 and Class2, or change to None to train on all 10 classes
classes = ['Class1', 'Class2'] 

print("Loading datasets...")
train_dataset = DAGMDataset(root_dir=root_dir, classes=classes, split='Train')
test_dataset = DAGMDataset(root_dir=root_dir, classes=classes, split='Test')

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")

In [ ]:
# Model definition (U-Net with ResNet34 encoder)
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(device)

criterion = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
best_iou = 0.0

print("Starting training...")
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
    train_loss = train_loss / len(train_loader.dataset)
    
    # Validation
    model.eval()
    val_loss = 0.0
    all_metrics = {'accuracy': 0, 'precision': 0, 'recall': 0, 'f1': 0, 'iou': 0}
    num_batches = len(test_loader)
    
    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item() * images.size(0)
            
            preds = torch.sigmoid(outputs)
            batch_metrics = calculate_metrics(preds, masks)
            for k in all_metrics:
                all_metrics[k] += batch_metrics[k]
                
    val_loss = val_loss / len(test_loader.dataset)
    for k in all_metrics:
        all_metrics[k] /= num_batches
        
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f} - Val IoU: {all_metrics['iou']:.4f}")
    
    if all_metrics['iou'] > best_iou:
        best_iou = all_metrics['iou']
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"Saved new best model with IoU: {best_iou:.4f}")

print("Training completed.")
print("Final Validation Metrics:")
for k, v in all_metrics.items():
    print(f"{k.capitalize()}: {v:.4f}")
    
with open('metrics.json', 'w') as f:
    json.dump(all_metrics, f)

print("\nNow you can download 'best_model.pth' and 'metrics.json' to use with your local GUI!")